# Retail Loan Credit Risk Workflow

This notebook is a public workflow reconstruction based on the project reports in this folder. The original modeling notebook was not found in the local scan, so this file is labeled as a reviewable reconstruction rather than the original source artifact.

## Objective

Prepare LendingClub loan data, segment borrowers, and train a charge-off screening model using only origination-time information.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [ ]:
DATA_PATH = Path("../data/raw/loan_data_2007_2014.csv")
TARGET_STATUSES = {"Fully Paid": 0, "Charged Off": 1}

raw = pd.read_csv(DATA_PATH, low_memory=False)
loans = raw[raw["loan_status"].isin(TARGET_STATUSES)].copy()
loans["charged_off"] = loans["loan_status"].map(TARGET_STATUSES)
loans.shape

In [ ]:
def clean_percent(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.replace("%", "", regex=False), errors="coerce") / 100


def clean_employment_length(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.replace("+ years", "", regex=False).str.replace("< 1 year", "0", regex=False)
    cleaned = cleaned.str.extract(r"(\d+)")[0]
    return pd.to_numeric(cleaned, errors="coerce")

loans["int_rate"] = clean_percent(loans["int_rate"])
loans["revol_util"] = clean_percent(loans["revol_util"])
loans["emp_length_num"] = clean_employment_length(loans["emp_length"])
loans["issue_d"] = pd.to_datetime(loans["issue_d"], errors="coerce")
loans["earliest_cr_line"] = pd.to_datetime(loans["earliest_cr_line"], errors="coerce")
loans["credit_history_years"] = (loans["issue_d"] - loans["earliest_cr_line"]).dt.days / 365.25

In [ ]:
numeric_features = [
    "loan_amnt", "term", "int_rate", "installment", "annual_inc", "dti",
    "revol_bal", "revol_util", "emp_length_num", "credit_history_years",
]
categorical_features = ["grade", "home_ownership", "verification_status", "purpose", "initial_list_status"]

model_data = loans[numeric_features + categorical_features + ["charged_off"]].copy()
model_data["term"] = model_data["term"].astype(str).str.extract(r"(\d+)").astype(float)

X = model_data[numeric_features + categorical_features]
y = model_data["charged_off"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.25, random_state=42)

In [ ]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

rf_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=25, class_weight="balanced", random_state=42, n_jobs=-1)),
])
rf_model.fit(X_train, y_train)
prob = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
threshold = 0.25
pred = (prob >= threshold).astype(int)
print(classification_report(y_test, pred, target_names=["fully_paid", "charged_off"]))
print(confusion_matrix(y_test, pred))

In [ ]:
segment_features = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "revol_bal", "revol_util", "credit_history_years"]
segment_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=5, random_state=42)),
    ("cluster", KMeans(n_clusters=4, random_state=42, n_init="auto")),
])
loans["segment"] = segment_pipeline.fit_predict(loans[segment_features])
loans.groupby("segment")[segment_features + ["charged_off"]].mean().round(3)

## Notes For Reviewers

The published reports contain the final results. This notebook shows the reproducible structure a reviewer would expect: cleaning, feature engineering, segmentation, imbalanced classification, and threshold tuning.